## 1. Setup and Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Code")
DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")

# Input: Cleaned StockTwits data by year
STOCKTWITS_FOLDER = DATA_DIR / "cleaned_by_year_mlcrowd"

# Intermediate: Exploded data (one row per symbol)
EXPLODED_FOLDER = DATA_DIR / "exploded_by_year_mlcrowd"
EXPLODED_FOLDER.mkdir(parents=True, exist_ok=True)

# CRSP data location
CRSP_FOLDER = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\CRSP")

# Output: Merged data
OUTPUT_FOLDER = DATA_DIR / "merged_with_crsp_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# =============================================================================
# PARAMETERS
# =============================================================================
# Horizons for abnormal returns (matching feature_proposal.md: 5, 21, 63, 252 days)
# Also including 1, 3, 10, 42 for more granularity
HORIZONS = [1, 3, 5, 21, 63]

print(f"StockTwits folder: {STOCKTWITS_FOLDER}")
print(f"Exploded folder: {EXPLODED_FOLDER}")
print(f"CRSP folder: {CRSP_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Horizons: {HORIZONS}")

## 2. Inspect Available Data

In [ ]:
# Check available StockTwits files
stocktwits_files = sorted([f for f in os.listdir(STOCKTWITS_FOLDER) if f.endswith('.csv')])
print(f"StockTwits files found: {len(stocktwits_files)}")
print(f"Files: {stocktwits_files}\n")

# Extract years from filenames
stocktwits_years = [int(f.split('_')[-1].replace('.csv', '')) for f in stocktwits_files]
print(f"Years in StockTwits data: {stocktwits_years}\n")

# Check available CRSP files
crsp_files = sorted([f for f in os.listdir(CRSP_FOLDER) if f.startswith('dsf_final_') and f.endswith('.pkl')])
print(f"CRSP files found: {len(crsp_files)}")
print(f"Files: {crsp_files}\n")

# Extract years from CRSP filenames
crsp_years = [int(f.split('_')[-1].replace('.pkl', '')) for f in crsp_files]
print(f"Years in CRSP data: {crsp_years}\n")

# Find overlapping years
overlap_years = sorted(set(stocktwits_years) & set(crsp_years))
print(f"Years available in both datasets: {overlap_years}")

## 2a. Inspect Sample Data Structure (Before Explosion)

## 3. Explode Symbol List

Before merging with CRSP, we need to explode the `symbol_list` column so that each symbol gets its own row. If a message mentions multiple symbols, it will create one row per symbol with all other columns duplicated.


In [ ]:
import ast

def explode_symbol_list(df):
    """
    Explode symbol_list column to create one row per symbol.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'symbol_list' column containing list of symbols
        
    Returns:
    --------
    pd.DataFrame: Exploded dataframe with 'symbol' column and duplicated other columns
    """
    df = df.copy()
    
    # Parse symbol_list if it's a string
    df['symbol_list_parsed'] = df['symbol_list'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    
    # Filter out rows with empty or invalid symbol lists
    df = df[df['symbol_list_parsed'].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )]
    
    # Explode the list - creates one row per symbol
    df_exploded = df.explode('symbol_list_parsed')
    
    # Rename the exploded column to 'symbol'
    df_exploded = df_exploded.rename(columns={'symbol_list_parsed': 'symbol'})
    
    # Drop the original symbol_list column
    df_exploded = df_exploded.drop(columns=['symbol_list'])
    
    # Reset index
    df_exploded = df_exploded.reset_index(drop=True)
    
    return df_exploded

In [ ]:
# Test the explosion function on sample data
if stocktwits_files:
    sample_file = STOCKTWITS_FOLDER / stocktwits_files[0]
    df_test = pd.read_csv(sample_file, nrows=1000)
    
    print(f"Original data shape: {df_test.shape}")
    print(f"Sample symbol_list values:")
    print(df_test['symbol_list'].head(10))
    
    # Check how many symbols per message
    df_test['symbol_list_parsed'] = df_test['symbol_list'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df_test['num_symbols'] = df_test['symbol_list_parsed'].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    
    print(f"\nSymbols per message distribution:")
    print(df_test['num_symbols'].value_counts().sort_index())
    
    # Test explosion
    df_exploded_test = explode_symbol_list(df_test.drop(columns=['symbol_list_parsed', 'num_symbols']))
    
    print(f"\nAfter explosion shape: {df_exploded_test.shape}")
    print(f"Expansion ratio: {len(df_exploded_test) / len(df_test):.2f}x")
    print(f"\nSample exploded data:")
    display(df_exploded_test.head(10))
    
    print(f"\nColumns after explosion: {list(df_exploded_test.columns)}")


In [ ]:
# Process all years and create exploded datasets
print(f"{'='*60}")
print(f"Exploding symbol_list for all years...")
print(f"{'='*60}\n")

explosion_stats = []

for year in tqdm(stocktwits_years, desc="Exploding data"):
    try:
        # Load file
        input_file = STOCKTWITS_FOLDER / f"stocktwits_cleaned_{year}.csv"
        df = pd.read_csv(input_file)
        
        original_rows = len(df)
        
        # Explode symbol list
        df_exploded = explode_symbol_list(df)
        
        exploded_rows = len(df_exploded)
        
        # Save exploded data
        output_file = EXPLODED_FOLDER / f"stocktwits_exploded_{year}.csv"
        df_exploded.to_csv(output_file, index=False)
        
        # Track statistics
        explosion_stats.append({
            'year': year,
            'original_rows': original_rows,
            'exploded_rows': exploded_rows,
            'expansion_ratio': exploded_rows / original_rows if original_rows > 0 else 0,
            'unique_symbols': df_exploded['symbol'].nunique()
        })
        
        print(f"  {year}: {original_rows:,} -> {exploded_rows:,} rows ({exploded_rows/original_rows:.2f}x), {df_exploded['symbol'].nunique():,} unique symbols")
        
    except Exception as e:
        print(f"  {year}: ERROR - {str(e)}")

print(f"\n{'='*60}")
print("Explosion Complete!")
print(f"{'='*60}")

# Summary
df_explosion_stats = pd.DataFrame(explosion_stats)
print(f"\nTotal original rows: {df_explosion_stats['original_rows'].sum():,}")
print(f"Total exploded rows: {df_explosion_stats['exploded_rows'].sum():,}")
print(f"Overall expansion ratio: {df_explosion_stats['exploded_rows'].sum() / df_explosion_stats['original_rows'].sum():.2f}x")
print(f"\nExploded files saved to: {EXPLODED_FOLDER}")


In [ ]:
# Load a sample StockTwits file
if stocktwits_files:
    sample_st_file = STOCKTWITS_FOLDER / stocktwits_files[0]
    df_st_sample = pd.read_csv(sample_st_file, nrows=1000)
    
    print(f"Sample StockTwits file: {stocktwits_files[0]}")
    print(f"Shape: {df_st_sample.shape}")
    print(f"\nColumns: {list(df_st_sample.columns)}")
    print(f"\nSample data:")
    display(df_st_sample.head())
    
    # Check if we need to extract symbol from symbol_list
    if 'symbol' in df_st_sample.columns:
        print(f"\nUnique symbols (sample): {df_st_sample['symbol'].nunique():,}")
        print(f"Top 10 symbols by volume:")
        print(df_st_sample['symbol'].value_counts().head(10))
    elif 'symbol_list' in df_st_sample.columns:
        print(f"\nNote: Data has 'symbol_list' column - needs to be parsed")
        print(f"Sample symbol_list values:")
        print(df_st_sample['symbol_list'].head())


In [ ]:
# Load a sample CRSP file
if crsp_files:
    sample_crsp_file = CRSP_FOLDER / crsp_files[0]
    df_crsp_sample = pd.read_pickle(sample_crsp_file)
    
    print(f"Sample CRSP file: {crsp_files[0]}")
    print(f"Shape: {df_crsp_sample.shape}")
    print(f"\nColumns: {list(df_crsp_sample.columns)}")
    print(f"\nSample data:")
    display(df_crsp_sample.head())
    
    print(f"\nUnique tickers in CRSP: {df_crsp_sample['ticker'].nunique():,}")
    print(f"Unique PERMNOs in CRSP: {df_crsp_sample['permno'].nunique():,}")

## 4. Define Merge Function

In [ ]:
def merge_stocktwits_with_crsp(df_stocktwits, df_crsp, horizons=HORIZONS):
    """
    Merge StockTwits data with CRSP data to filter for US-listed stocks.
    Also calculates abnormal returns for multiple models and horizons.
    
    Parameters:
    -----------
    df_stocktwits : pd.DataFrame
        Exploded StockTwits data with 'symbol' column (one row per symbol)
    df_crsp : pd.DataFrame
        CRSP data with columns: permno, ticker, date, returns, and abnormal return components
    horizons : list
        List of horizons (in days) for abnormal return calculations
        
    Returns:
    --------
    pd.DataFrame: Merged data with US-listed stocks and abnormal returns
    """
    # Ensure date columns are datetime
    df_stocktwits = df_stocktwits.copy()
    df_crsp = df_crsp.copy()
    
    df_stocktwits['date'] = pd.to_datetime(df_stocktwits['date'])
    df_crsp['date'] = pd.to_datetime(df_crsp['date'])
    
    # Standardize ticker format (uppercase)
    df_crsp['ticker'] = df_crsp['ticker'].str.upper()
    df_stocktwits['symbol'] = df_stocktwits['symbol'].str.upper()
    
    # --- Calculate Abnormal Returns ---
    print("  Calculating abnormal returns...")
    
    # 1. DGTW: Abnormal return = cumulative return - benchmark return
    for h in horizons:
        cumret_col = f'f_cumret{h}'
        dgtw_col = f'f_dgtw_ret{h}'
        if cumret_col in df_crsp.columns and dgtw_col in df_crsp.columns:
            df_crsp[f'ar_dgtw_{h}'] = df_crsp[cumret_col] - df_crsp[dgtw_col]
    
    # 2. CAPM, FF3, FF5, FF6: Cumulative abnormal returns
    # Sum individual abnormal returns from day 1 to day h
    models = ['capm', 'FF3', 'FF5', 'FF6']
    
    for model in models:
        # Find max horizon to check what columns are available
        max_horizon = max(horizons)
        ar_cols_available = []
        for i in range(1, max_horizon + 1):
            col = f'f_{model}_ar{i}'
            if col in df_crsp.columns:
                ar_cols_available.append(col)
        
        # Calculate cumulative abnormal returns for each horizon
        if ar_cols_available:  # Only if we have the columns
            for h in horizons:
                # Sum from day 1 to day h
                cols_to_sum = [f'f_{model}_ar{i}' for i in range(1, h + 1) 
                              if f'f_{model}_ar{i}' in df_crsp.columns]
                if cols_to_sum:
                    df_crsp[f'ar_{model}_{h}'] = df_crsp[cols_to_sum].sum(axis=1)
    
    # --- Select Columns for Merge ---
    crsp_cols = ['permno', 'ticker', 'date']
    
    # Add market data columns
    market_cols = ['prc', 'vol', 'shrout', 'ret']
    for col in market_cols:
        if col in df_crsp.columns:
            crsp_cols.append(col)
    
    # Add all calculated abnormal return columns
    for model in ['dgtw', 'capm', 'FF3', 'FF5', 'FF6']:
        for h in horizons:
            ar_col = f'ar_{model}_{h}'
            if ar_col in df_crsp.columns:
                crsp_cols.append(ar_col)
    
    df_crsp_subset = df_crsp[crsp_cols].copy()
    
    # Remove duplicates in CRSP (ticker-date level)
    # Keep the first occurrence if there are multiple PERMNOs for the same ticker-date
    df_crsp_subset = df_crsp_subset.drop_duplicates(subset=['ticker', 'date'], keep='first')
    
    # --- Merge StockTwits with CRSP ---
    df_merged = pd.merge(
        df_stocktwits,
        df_crsp_subset,
        left_on=['symbol', 'date'],
        right_on=['ticker', 'date'],
        how='inner'  # Keep only matches (US-listed stocks)
    )
    
    # Drop redundant ticker column (we already have symbol)
    if 'ticker' in df_merged.columns:
        df_merged = df_merged.drop(columns=['ticker'])
    
    return df_merged


## 5. Test Merge on Sample Year

In [ ]:
# Test on first overlapping year
if overlap_years:
    test_year = overlap_years[0]
    
    print(f"Testing merge on year: {test_year}\n")
    
    # Load test data (from exploded folder)
    st_file = EXPLODED_FOLDER / f"stocktwits_exploded_{test_year}.csv"
    crsp_file = CRSP_FOLDER / f"dsf_final_{test_year}.pkl"
    
    df_st_test = pd.read_csv(st_file)
    df_crsp_test = pd.read_pickle(crsp_file)
    
    print(f"StockTwits rows (exploded): {len(df_st_test):,}")
    print(f"CRSP rows: {len(df_crsp_test):,}")
    print(f"Unique symbols in StockTwits: {df_st_test['symbol'].nunique():,}")
    print(f"Unique tickers in CRSP: {df_crsp_test['ticker'].nunique():,}\n")
    
    # Perform merge
    df_merged_test = merge_stocktwits_with_crsp(df_st_test, df_crsp_test)
    
    print(f"\nMerged rows: {len(df_merged_test):,}")
    print(f"Match rate: {len(df_merged_test) / len(df_st_test) * 100:.2f}%")
    print(f"Unique symbols matched: {df_merged_test['symbol'].nunique():,}")
    print(f"\nMerged columns: {list(df_merged_test.columns)}")
    print(f"\nSample merged data:")
    display(df_merged_test.head(10))
else:
    print("No overlapping years found between StockTwits and CRSP data.")


## 6. Process All Years (Merge with CRSP)

In [ ]:
# Process all overlapping years
merge_stats = []

print(f"{'='*60}")
print(f"Starting year-by-year merge with CRSP...")
print(f"{'='*60}\n")

for year in tqdm(overlap_years, desc="Processing years"):
    try:
        # Load files (from exploded folder)
        st_file = EXPLODED_FOLDER / f"stocktwits_exploded_{year}.csv"
        crsp_file = CRSP_FOLDER / f"dsf_final_{year}.pkl"
        
        df_st = pd.read_csv(st_file)
        df_crsp = pd.read_pickle(crsp_file)
        
        # Merge
        df_merged = merge_stocktwits_with_crsp(df_st, df_crsp)
        
        # Save merged data
        output_file = OUTPUT_FOLDER / f"stocktwits_crsp_{year}.csv"
        df_merged.to_csv(output_file, index=False)
        
        # Track statistics
        match_rate = (len(df_merged) / len(df_st)) * 100 if len(df_st) > 0 else 0
        
        # Count abnormal return columns added
        ar_cols_in_merge = [col for col in df_merged.columns if col.startswith('ar_')]
        
        merge_stats.append({
            'year': year,
            'stocktwits_rows': len(df_st),
            'stocktwits_symbols': df_st['symbol'].nunique(),
            'crsp_rows': len(df_crsp),
            'crsp_tickers': df_crsp['ticker'].nunique(),
            'merged_rows': len(df_merged),
            'merged_symbols': df_merged['symbol'].nunique(),
            'match_rate': match_rate,
            'ar_columns': len(ar_cols_in_merge)
        })
        
        print(f"  {year}: {len(df_st):,} rows -> {len(df_merged):,} matched ({match_rate:.1f}%), {df_merged['symbol'].nunique():,} symbols, {len(ar_cols_in_merge)} AR cols")
        
    except Exception as e:
        print(f"  {year}: ERROR - {str(e)}")

print(f"\n{'='*60}")
print("Merge Complete!")
print(f"{'='*60}")


## 7. Merge Summary Statistics

In [ ]:
# Display summary statistics
df_stats = pd.DataFrame(merge_stats)

print(f"\n{'='*60}")
print("MERGE SUMMARY STATISTICS")
print(f"{'='*60}\n")

print(df_stats.to_string(index=False))

print(f"\n{'='*60}")
print("OVERALL STATISTICS")
print(f"{'='*60}")
print(f"Total StockTwits rows: {df_stats['stocktwits_rows'].sum():,}")
print(f"Total merged rows: {df_stats['merged_rows'].sum():,}")
if df_stats['stocktwits_rows'].sum() > 0:
    print(f"Overall match rate: {(df_stats['merged_rows'].sum() / df_stats['stocktwits_rows'].sum() * 100):.2f}%")

print(f"\nAverage symbols per year:")
print(f"  StockTwits: {df_stats['stocktwits_symbols'].mean():.0f}")
print(f"  After CRSP filter: {df_stats['merged_symbols'].mean():.0f}")
print(f"  Symbol retention: {(df_stats['merged_symbols'].mean() / df_stats['stocktwits_symbols'].mean() * 100):.1f}%")

print(f"\nOutput folder: {OUTPUT_FOLDER}")

## 8. Analyze Merge Match Quality

In [ ]:
# Visualize match rates by year
import matplotlib.pyplot as plt

if len(df_stats) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Plot 1: Match rate by year
    axes[0].plot(df_stats['year'], df_stats['match_rate'], marker='o', linewidth=2)
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Match Rate (%)')
    axes[0].set_title('StockTwits-CRSP Match Rate by Year')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 100])
    
    # Plot 2: Volume by year
    axes[1].bar(df_stats['year'], df_stats['stocktwits_rows'], alpha=0.5, label='Original StockTwits')
    axes[1].bar(df_stats['year'], df_stats['merged_rows'], alpha=0.7, label='After CRSP Filter')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Number of Messages')
    axes[1].set_title('Message Volume: Before and After CRSP Filter')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMatch rate statistics:")
    print(f"  Mean: {df_stats['match_rate'].mean():.2f}%")
    print(f"  Median: {df_stats['match_rate'].median():.2f}%")
    print(f"  Min: {df_stats['match_rate'].min():.2f}% (year {df_stats.loc[df_stats['match_rate'].idxmin(), 'year']:.0f})")
    print(f"  Max: {df_stats['match_rate'].max():.2f}% (year {df_stats.loc[df_stats['match_rate'].idxmax(), 'year']:.0f})")

## 9. Identify Unmatched Symbols

In [ ]:
# Load one year to analyze which symbols didn't match
if overlap_years:
    analysis_year = overlap_years[-1]  # Use most recent year
    
    st_file = EXPLODED_FOLDER / f"stocktwits_exploded_{analysis_year}.csv"
    crsp_file = CRSP_FOLDER / f"dsf_final_{analysis_year}.pkl"
    merged_file = OUTPUT_FOLDER / f"stocktwits_crsp_{analysis_year}.csv"
    
    df_st_analysis = pd.read_csv(st_file)
    df_crsp_analysis = pd.read_pickle(crsp_file)
    df_merged_analysis = pd.read_csv(merged_file)
    
    # Find symbols in StockTwits but not matched with CRSP
    st_symbols = set(df_st_analysis['symbol'].str.upper())
    crsp_tickers = set(df_crsp_analysis['ticker'].str.upper())
    matched_symbols = set(df_merged_analysis['symbol'].str.upper())
    unmatched_symbols = st_symbols - matched_symbols
    
    print(f"\nAnalyzing unmatched symbols for year {analysis_year}:")
    print(f"{'='*60}")
    print(f"Total symbols in StockTwits: {len(st_symbols):,}")
    print(f"Total tickers in CRSP: {len(crsp_tickers):,}")
    print(f"Matched symbols: {len(matched_symbols):,}")
    print(f"Unmatched symbols: {len(unmatched_symbols):,}\n")
    
    # Count messages for unmatched symbols
    unmatched_df = df_st_analysis[df_st_analysis['symbol'].str.upper().isin(unmatched_symbols)]
    unmatched_volume = unmatched_df['symbol'].value_counts().head(20)
    
    print(f"Top 20 unmatched symbols by message volume:")
    print(unmatched_volume)
    
    # Check if they're in CRSP but different dates
    print(f"\nChecking if unmatched symbols exist in CRSP (different dates):")
    top_unmatched = unmatched_volume.head(5).index.tolist()
    for sym in top_unmatched:
        in_crsp = sym in crsp_tickers
        print(f"  {sym}: {'YES (in CRSP)' if in_crsp else 'NO (not in CRSP)'}")


## 10. Final Data Schema

The merged data includes:

### From StockTwits (cleaned & exploded):
- `message_id`: Unique message identifier
- `user_id`: User identifier
- `symbol`: Stock ticker symbol (one row per symbol)
- `date`: Trading date (first market close after message)
- `created_at`: Original timestamp (US/Eastern)
- `sentiment`: Bullish/Bearish label
- `time`: Time of day
- `hour`: Hour of day (0-23)
- `is_after_hours`: After market close indicator
- `session`: Market session category

### From CRSP - Market Data:
- `permno`: CRSP permanent number (stock identifier)
- `prc`: Closing price
- `vol`: Trading volume
- `shrout`: Shares outstanding
- `ret`: Daily return

### From CRSP - Abnormal Returns (Target Variables):
Multiple models and horizons for machine learning prediction:

**Models:**
- `ar_dgtw_{h}`: DGTW (Daniel, Grinblatt, Titman, Wermers) abnormal return
- `ar_capm_{h}`: CAPM abnormal return
- `ar_FF3_{h}`: Fama-French 3-factor abnormal return
- `ar_FF5_{h}`: Fama-French 5-factor abnormal return
- `ar_FF6_{h}`: Fama-French 6-factor abnormal return

**Horizons (h):** 1, 3, 5, 10, 21, 42, 63, 252 trading days
- Standardized horizons from feature_proposal.md: 5, 21, 63, 252 days (~1 week, 1 month, 1 quarter, 1 year)
- Additional horizons for granularity: 1, 3, 10, 42 days

### Key Notes:
- **Inner join**: Only stocks that appear in CRSP (US-listed) are retained
- **Date matching**: Ensures message date aligns with CRSP trading date
- **Target variables ready**: Abnormal returns serve as dependent variables for ML models
- **Feature extraction ready**: Data is filtered and ready for calculating social media features from feature_proposal.md